# Task 2.1 — Annotation Dataset Creation

Build a balanced 5,000-sample SFT dataset from scored factors for fine-tuning
DeepSeek-R1-Distill-Qwen-14B on financial sentiment classification.

| | |
|---|---|
| **Input** | `../task_1/output/factors_scored/{TICKER}/*_factors.json` |
| **Input** | `../task_1/ticker_mapping.json` |
| **Output** | `data/sft_dataset.jsonl` — full 5K dataset |
| **Output** | `data/sft_train.jsonl` — 80% training split (~4,000) |
| **Output** | `data/sft_val.jsonl` — 20% validation split (~1,000) |
| **Output** | `data/dataset_stats.json` — distribution statistics |

> No GPU required. Runs locally or on any ACCRE node.

## Step 1: Imports & Configuration

In [ ]:
import json
import logging
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
TASK1_DIR = _cwd.parent / "task_1" if (_cwd.parent / "task_1").exists() else _cwd / ".." / "task_1"
TASK1_DIR = TASK1_DIR.resolve()

FACTORS_SCORED_DIR = TASK1_DIR / "output" / "factors_scored"
TICKER_MAPPING_PATH = TASK1_DIR / "ticker_mapping.json"
OUTPUT_DIR = _cwd / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────
TARGET_DATASET_SIZE = 5000
TARGET_PER_CLASS = 1000  # 5 classes × 1000 = 5000
MIN_CONFIDENCE = 0.5
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]

# ── Load sub-sector mapping ───────────────────────────────────────────────
with open(TICKER_MAPPING_PATH) as f:
    _ticker_map = json.load(f)

TICKER_SUBSECTOR = {}
for subsector, tickers in _ticker_map.items():
    for t in tickers:
        TICKER_SUBSECTOR[t] = subsector

# ── Logging ───────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_2_1")

print(f"Task 1 dir:             {TASK1_DIR}")
print(f"Scored factors dir:     {FACTORS_SCORED_DIR} (exists={FACTORS_SCORED_DIR.exists()})")
print(f"Ticker mapping:         {TICKER_MAPPING_PATH} (exists={TICKER_MAPPING_PATH.exists()})")
print(f"Output dir:             {OUTPUT_DIR}")
print(f"Sub-sectors:            {list(_ticker_map.keys())}")
print(f"Total tickers:          {len(TICKER_SUBSECTOR)}")
print(f"Target dataset size:    {TARGET_DATASET_SIZE}")
print(f"Min confidence:         {MIN_CONFIDENCE}")

## Step 2: Load All Scored Factors

Walk `output/factors_scored/{TICKER}/` and build a flat DataFrame. Each row = one factor with its sentiment label.

In [ ]:
rows = []
skipped_no_sentiment = 0
files_loaded = 0

for ticker_dir in sorted(FACTORS_SCORED_DIR.iterdir()):
    if not ticker_dir.is_dir():
        continue
    ticker = ticker_dir.name
    subsector = TICKER_SUBSECTOR.get(ticker, "unknown")

    for fpath in sorted(ticker_dir.glob("*_factors.json")):
        with open(fpath) as f:
            data = json.load(f)

        files_loaded += 1
        form = data.get("form", "")
        filing_date = data.get("filing_date", "")

        for fac in data.get("factors", []):
            sentiment = fac.get("sentiment")
            if not isinstance(sentiment, dict) or not sentiment.get("label"):
                skipped_no_sentiment += 1
                continue

            # Flatten evidence array into a single text string
            evidence_parts = []
            for ev in fac.get("evidence", []):
                text = ev.get("text", "").strip()
                if text:
                    section = ev.get("section", "")
                    subsection = ev.get("subsection", "")
                    source = f" [{section}/{subsection}]" if section else ""
                    evidence_parts.append(f"{text}{source}")
            evidence_text = " | ".join(evidence_parts) if evidence_parts else ""

            rows.append({
                "ticker": ticker,
                "sub_sector": subsector,
                "form": form,
                "filing_date": filing_date,
                "key": fac.get("key", ""),
                "category": fac.get("category", ""),
                "summary": fac.get("summary", ""),
                "evidence_text": evidence_text,
                "label": sentiment["label"],
                "rationale": sentiment.get("rationale", ""),
                "confidence": sentiment.get("confidence", 0.0),
            })

all_factors_df = pd.DataFrame(rows)
all_factors_df["filing_date"] = pd.to_datetime(all_factors_df["filing_date"])
all_factors_df["year"] = all_factors_df["filing_date"].dt.year

print(f"Files loaded: {files_loaded}")
print(f"Total factors with sentiment: {len(all_factors_df)}")
print(f"Skipped (no sentiment): {skipped_no_sentiment}")
print(f"Unique tickers: {all_factors_df['ticker'].nunique()}")
print(f"Unique filings: {all_factors_df.groupby(['ticker', 'filing_date']).ngroups}")
print(f"Unique factor keys: {all_factors_df['key'].nunique()}")
print(f"Unique categories: {all_factors_df['category'].nunique()}")
print(f"\nLabel distribution (raw):")
print(all_factors_df["label"].value_counts().reindex(LABEL_ORDER).to_string())
all_factors_df.head(5)

## Step 3: Quality Filter

Drop factors with low confidence or empty rationale — these are unreliable annotations.

In [ ]:
before_count = len(all_factors_df)

# Filter: confidence >= threshold AND rationale is non-empty
mask_confidence = all_factors_df["confidence"] >= MIN_CONFIDENCE
mask_rationale = all_factors_df["rationale"].str.strip().str.len() > 0
mask_summary = all_factors_df["summary"].str.strip().str.len() > 0
mask_valid_label = all_factors_df["label"].isin(LABEL_ORDER)

filtered_df = all_factors_df[mask_confidence & mask_rationale & mask_summary & mask_valid_label].copy()

dropped = before_count - len(filtered_df)
print(f"Before filter: {before_count}")
print(f"After filter:  {len(filtered_df)}")
print(f"Dropped:       {dropped} ({dropped/before_count:.1%})")
print(f"\nDrop reasons:")
print(f"  Low confidence (<{MIN_CONFIDENCE}): {(~mask_confidence).sum()}")
print(f"  Empty rationale:                    {(~mask_rationale).sum()}")
print(f"  Empty summary:                      {(~mask_summary).sum()}")
print(f"  Invalid label:                      {(~mask_valid_label).sum()}")
print(f"\nFiltered label distribution:")
print(filtered_df["label"].value_counts().reindex(LABEL_ORDER).to_string())
print(f"\nFiltered sub-sector distribution:")
print(filtered_df["sub_sector"].value_counts().to_string())

## Step 4: Label Distribution Analysis

Understand the data before sampling: label × sub-sector × category × year.

In [ ]:
print("="*60)
print("LABEL × SUB-SECTOR")
print("="*60)
cross_tab = pd.crosstab(filtered_df["sub_sector"], filtered_df["label"])
cross_tab = cross_tab.reindex(columns=LABEL_ORDER, fill_value=0)
print(cross_tab.to_string())
print(f"\nTotal per sub-sector: {cross_tab.sum(axis=1).to_dict()}")

print(f"\n{'='*60}")
print("LABEL × CATEGORY (top 14)")
print("="*60)
cat_cross = pd.crosstab(filtered_df["category"], filtered_df["label"])
cat_cross = cat_cross.reindex(columns=LABEL_ORDER, fill_value=0)
cat_cross["total"] = cat_cross.sum(axis=1)
cat_cross = cat_cross.sort_values("total", ascending=False)
print(cat_cross.to_string())

print(f"\n{'='*60}")
print("LABEL × YEAR")
print("="*60)
year_cross = pd.crosstab(filtered_df["year"], filtered_df["label"])
year_cross = year_cross.reindex(columns=LABEL_ORDER, fill_value=0)
print(year_cross.to_string())

print(f"\n{'='*60}")
print("CONFIDENCE STATS BY LABEL")
print("="*60)
conf_stats = filtered_df.groupby("label")["confidence"].describe()
print(conf_stats.reindex(LABEL_ORDER).to_string())

## Step 5: Stratified Sampling

Select 5,000 data points balanced across:
1. ~1,000 per sentiment class
2. Proportional sub-sector representation within each class
3. All 14 categories represented
4. Spread across filing years

Strategy: sample up to 1,000 per label. Within each label, sample proportionally to sub-sector size. If a label has fewer than 1,000, take all of them.

In [ ]:
np.random.seed(RANDOM_SEED)

sampled_parts = []

for label in LABEL_ORDER:
    label_df = filtered_df[filtered_df["label"] == label]
    available = len(label_df)
    target = min(TARGET_PER_CLASS, available)

    if available <= target:
        # Take all — this class is underrepresented
        sampled_parts.append(label_df)
        print(f"{label}: taking ALL {available} (target was {TARGET_PER_CLASS})")
    else:
        # Stratified sample within this label: proportional to sub-sector size
        # Group by sub-sector, allocate proportionally
        subsector_counts = label_df["sub_sector"].value_counts()
        subsector_fracs = subsector_counts / subsector_counts.sum()

        label_samples = []
        remaining = target

        for subsector in subsector_fracs.index:
            ss_df = label_df[label_df["sub_sector"] == subsector]
            # Proportional allocation, at least 1 if subsector exists
            n_alloc = max(1, int(round(subsector_fracs[subsector] * target)))
            n_alloc = min(n_alloc, len(ss_df), remaining)
            if n_alloc > 0:
                label_samples.append(ss_df.sample(n=n_alloc, random_state=RANDOM_SEED))
                remaining -= n_alloc

        # If we're short due to rounding, sample more from the largest sub-sector
        if remaining > 0:
            already_sampled_idx = pd.concat(label_samples).index
            leftover = label_df.drop(already_sampled_idx)
            if len(leftover) >= remaining:
                label_samples.append(leftover.sample(n=remaining, random_state=RANDOM_SEED))
            else:
                label_samples.append(leftover)

        label_sampled = pd.concat(label_samples)
        sampled_parts.append(label_sampled)
        print(f"{label}: sampled {len(label_sampled)} / {available} "
              f"(sub-sectors: {label_sampled['sub_sector'].value_counts().to_dict()})")

sampled_df = pd.concat(sampled_parts).drop_duplicates().reset_index(drop=True)

print(f"\nTotal sampled: {len(sampled_df)}")
print(f"\nFinal label distribution:")
print(sampled_df["label"].value_counts().reindex(LABEL_ORDER).to_string())
print(f"\nFinal sub-sector distribution:")
print(sampled_df["sub_sector"].value_counts().to_string())
print(f"\nCategories covered: {sampled_df['category'].nunique()} / {filtered_df['category'].nunique()}")
print(f"Years covered: {sorted(sampled_df['year'].unique())}")

## Step 6: Format as Chat JSONL

Convert each sample into `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`
format compatible with `trl` SFTTrainer.

In [ ]:
SYSTEM_PROMPT = (
    "You are a financial analyst specializing in SEC filing analysis. "
    "Classify the sentiment of the given factor and respond with JSON only."
)


def format_user_prompt(row: pd.Series) -> str:
    """Build the user prompt from a factor row."""
    evidence = row["evidence_text"]
    # Truncate evidence if too long (keep under ~1500 chars to leave room for output)
    if len(evidence) > 1500:
        evidence = evidence[:1500] + "..."

    return (
        f"Given the following factor extracted from a {row['form']} filing "
        f"for {row['ticker']} ({row['sub_sector']} sub-sector), classify the sentiment.\n\n"
        f"Category: {row['category']}\n"
        f"Factor: {row['key']}\n"
        f"Summary: {row['summary']}\n"
        f"Evidence: {evidence}\n\n"
        f"Classify the sentiment as one of: very_negative, negative, neutral, positive, very_positive.\n"
        f"Provide a brief rationale and confidence score (0.0-1.0).\n"
        f"Respond with JSON only."
    )


def format_assistant_response(row: pd.Series) -> str:
    """Build the expected assistant response."""
    return json.dumps({
        "label": row["label"],
        "rationale": row["rationale"],
        "confidence": round(float(row["confidence"]), 2),
    })


def format_chat_message(row: pd.Series) -> dict:
    """Convert a row to chat-format dict for SFT."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_user_prompt(row)},
            {"role": "assistant", "content": format_assistant_response(row)},
        ]
    }


# Convert all samples
chat_records = [format_chat_message(row) for _, row in sampled_df.iterrows()]

print(f"Formatted {len(chat_records)} chat records")
print(f"\nExample (first record):")
print(json.dumps(chat_records[0], indent=2)[:1500])

## Step 7: Train/Val Split

80/20 stratified split by label.

In [ ]:
# We need the labels aligned with chat_records for stratification
labels_for_split = sampled_df["label"].values

train_records, val_records, train_labels, val_labels = train_test_split(
    chat_records, labels_for_split,
    test_size=1 - TRAIN_RATIO,
    random_state=RANDOM_SEED,
    stratify=labels_for_split,
)

print(f"Train: {len(train_records)}")
print(f"Val:   {len(val_records)}")
print(f"\nTrain label distribution:")
train_label_counts = Counter(train_labels)
for label in LABEL_ORDER:
    print(f"  {label}: {train_label_counts.get(label, 0)}")
print(f"\nVal label distribution:")
val_label_counts = Counter(val_labels)
for label in LABEL_ORDER:
    print(f"  {label}: {val_label_counts.get(label, 0)}")

## Step 8: Save Files & Stats

In [ ]:
def write_jsonl(records: list, path: Path):
    """Write a list of dicts to a JSONL file."""
    with open(path, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")
    print(f"  Saved {len(records)} records to {path.name} ({path.stat().st_size / 1024:.1f} KB)")


# Save datasets
dataset_path = OUTPUT_DIR / "sft_dataset.jsonl"
train_path = OUTPUT_DIR / "sft_train.jsonl"
val_path = OUTPUT_DIR / "sft_val.jsonl"

write_jsonl(chat_records, dataset_path)
write_jsonl(train_records, train_path)
write_jsonl(val_records, val_path)

# Save distribution stats
stats = {
    "total_scored_factors": len(all_factors_df),
    "after_quality_filter": len(filtered_df),
    "dataset_size": len(chat_records),
    "train_size": len(train_records),
    "val_size": len(val_records),
    "label_distribution": sampled_df["label"].value_counts().reindex(LABEL_ORDER).to_dict(),
    "sub_sector_distribution": sampled_df["sub_sector"].value_counts().to_dict(),
    "category_distribution": sampled_df["category"].value_counts().to_dict(),
    "year_range": [int(sampled_df["year"].min()), int(sampled_df["year"].max())],
    "tickers_represented": int(sampled_df["ticker"].nunique()),
    "categories_represented": int(sampled_df["category"].nunique()),
    "min_confidence_threshold": MIN_CONFIDENCE,
    "avg_confidence": round(float(sampled_df["confidence"].mean()), 3),
    "train_label_distribution": {label: int(train_label_counts.get(label, 0)) for label in LABEL_ORDER},
    "val_label_distribution": {label: int(val_label_counts.get(label, 0)) for label in LABEL_ORDER},
}

stats_path = OUTPUT_DIR / "dataset_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"  Saved stats to {stats_path.name}")

print(f"\nAll files saved to {OUTPUT_DIR}/")

## Step 9: Visualizations

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

label_colors = ["#d62728", "#ff7f0e", "#999999", "#2ca02c", "#1f77b4"]

# ── Viz 1: Label distribution ─────────────────────────────────────────────
ax1 = axes[0]
label_counts = sampled_df["label"].value_counts().reindex(LABEL_ORDER)
bars = ax1.bar(range(5), label_counts.values, color=label_colors)
ax1.set_xticks(range(5))
ax1.set_xticklabels([l.replace("_", "\n") for l in LABEL_ORDER], fontsize=9)
ax1.set_ylabel("Count")
ax1.set_title(f"SFT Dataset — Label Distribution (N={len(sampled_df)})")
for bar, val in zip(bars, label_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")

# ── Viz 2: Sub-sector × Label heatmap ─────────────────────────────────────
ax2 = axes[1]
ss_label = pd.crosstab(sampled_df["sub_sector"], sampled_df["label"])
ss_label = ss_label.reindex(columns=LABEL_ORDER, fill_value=0)
sns.heatmap(ss_label, annot=True, fmt="d", cmap="YlOrRd", ax=ax2, cbar_kws={"label": "Count"})
ax2.set_title("Sub-Sector × Label")
ax2.set_xlabel("")
ax2.set_ylabel("")

# ── Viz 3: Category coverage ─────────────────────────────────────────────
ax3 = axes[2]
cat_counts = sampled_df["category"].value_counts().sort_values(ascending=True)
ax3.barh(range(len(cat_counts)), cat_counts.values, color="steelblue")
ax3.set_yticks(range(len(cat_counts)))
ax3.set_yticklabels(cat_counts.index, fontsize=8)
ax3.set_xlabel("Count")
ax3.set_title(f"Category Coverage ({len(cat_counts)} categories)")
for i, val in enumerate(cat_counts.values):
    ax3.text(val + 2, i, str(val), va="center", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nDataset ready for fine-tuning:")
print(f"  {train_path.name}: {len(train_records)} training samples")
print(f"  {val_path.name}: {len(val_records)} validation samples")